# Object Interaction and Active Objects

**Part I · Visualization** — Tutorial 10

Make scenes interactive with the pointer-event system, then use the
high-level `ActPoint` convenience class for common drag patterns. You will
learn to:

- Configure triggers with `InteractionConfig` / `InteractionTrigger` /
  `InteractionEventType` / `MouseButton` / `ModifierKey` / `DragMode`.
- Register handlers with `set_interaction()` / `on_interaction()`.
- Read the event dataclasses (`ClickEvent`, `DragEvent`, `ScrollEvent`) and the
  attached `Camera` (world↔screen `project()` / `unproject()`).
- Use `ActPoint` and its drag lifecycle (`handler`, `on_drag_start`,
  `on_drag_end`, `drag_mode`).


## Setup


In [ ]:
from pytanga.geometry import Point
from pytanga.viz import (
    ActPoint, DragMode, InteractionConfig, InteractionEventType, InteractionTrigger,
    ModifierKey, MouseButton, Visualizer,
)


## 1. Pointer-event system

`InteractionTrigger` defines *when* an event fires (event type, mouse button,
modifier keys, drag-mode constraint plane). `InteractionConfig` bundles a
trigger list per entity with throttling and optional hover feedback.


In [ ]:
# A left-click trigger:
click = InteractionTrigger(InteractionEventType.CLICK, mouse_button=MouseButton.LEFT)

# A Ctrl+Shift drag constrained to the YZ plane:
drag = InteractionTrigger(
    InteractionEventType.DRAG,
    mouse_button=MouseButton.LEFT,
    modifiers=frozenset({ModifierKey.CTRL, ModifierKey.SHIFT}),
    drag_mode=DragMode.YZ_PLANE,
)

config = InteractionConfig(
    enabled=True,
    triggers=[click, drag],
    throttle_ms=40,            # max rate for drag_move / scroll events
    hover_emissive="#ffff44",  # optional glow on hover
)
print("interaction config built:", config)


## 2. Registering handlers

`set_interaction(id, config)` attaches a config to an entity; `on_interaction(
id, event_type, handler)` registers an **async** handler for a specific event
type.


In [ ]:
viz = Visualizer(title="Interaction — handlers", add_default_axes=False, add_default_grid=False)

pid = viz.add(Point(0, 0, 2), color="#ff4444", label="drag me")

viz.set_interaction(
    pid,
    InteractionConfig(
        enabled=True,
        triggers=[
            InteractionTrigger(
                InteractionEventType.DRAG,
                mouse_button=MouseButton.LEFT,
                modifiers=frozenset({ModifierKey.SHIFT}),
                drag_mode=DragMode.XY_PLANE,
            ),
        ],
        throttle_ms=40,
    ),
)

async def on_drag(event):
    # event.world_position is a pytanga.geometry.Point
    viz.update_entity(event.object_id, event.world_position)
    viz.flush()

viz.on_interaction(pid, InteractionEventType.DRAG_MOVE, on_drag)
viz.flush()
viz.display_snapshot()


## 3. Event dataclasses and the attached `Camera`

Every event carries a `Camera` for world↔screen conversion:

- `ClickEvent` — `object_id`, `event_type`, `mouse_button`, `modifiers`,
  `screen_position`, `world_position` (Point), `world_normal` (Direction).
- `DragEvent` — adds `delta_pixels`, `world_delta`, `drag_mode`.
- `ScrollEvent` — adds `delta_xy`.

`camera.project(obj)` maps a world `Point`/`Direction` to screen pixels;
`camera.unproject(obj, depth)` maps back.


In [ ]:
# Inside an interaction handler:
#   px, py = event.camera.project(event.world_position)   # world → screen pixels
#   wp     = event.camera.unproject(Point(px, py), depth=5.0)  # screen → world

print("event.camera.project() / .unproject() are available on every event")


## 4. `ActPoint` — a self-registering draggable point

`ActPoint` creates a `Point` and registers its own interaction handlers. Its
visual style is set via `viz.add(ap, color=..., style=..., label=...)`, not on
the constructor. In 3D it registers four modifier-switched drag planes
(none = view plane, `Shift` = XY, `Ctrl` = XZ, `Ctrl+Shift` = YZ); in 2D the
unmodified drag defaults to the XY plane.


In [ ]:
viz = Visualizer(title="Interaction — ActPoint", add_default_axes=False, add_default_grid=False)

ap = ActPoint(Point(0, 0, 2))
viz.add(ap, color="#ff4444", label="P")

viz.flush()
viz.display_snapshot()


## 5. `ActPoint` drag lifecycle

The move-phase `handler` callback runs before the default movement; return
`False` to let `ActPoint` move the point and flush, or `True` to fully handle
it. `on_drag_start` / `on_drag_end` are pure notifications.


In [ ]:
async def on_move(event, ap):
    # ap.point is the current position; ap.viz_handle gives scene access.
    print("dragging to", event.world_position)
    return False   # let ActPoint move the point and flush

async def on_start(event, ap):
    print("drag started at", ap.point)

async def on_end(event, ap):
    print("drag ended at", ap.point)

viz = Visualizer(title="Interaction — lifecycle", add_default_axes=False, add_default_grid=False)
ap = ActPoint(
    Point(0, 0, 2),
    handler=on_move,
    on_drag_start=on_start,
    on_drag_end=on_end,
)
viz.add(ap, color="#ff4444", label="P")
viz.flush()
viz.display_snapshot()


## 6. `drag_mode` constraint

Pass `drag_mode=` to pin the unmodified left-button drag to a single plane. When
omitted, 2D visualizers default to `XY_PLANE` and 3D keeps the four
modifier-switched triggers.


In [ ]:
viz = Visualizer(title="Interaction — drag_mode", add_default_axes=False, add_default_grid=False)

# Constrain the unmodified drag to the XY plane (no modifier triggers):
ap = ActPoint(Point(1.0, 2.0, 0.0), drag_mode=DragMode.XY_PLANE)
viz.add(ap, color="#ff4444", label="P")

viz.flush()
viz.display_snapshot()


## 7. Labels

`ActPoint` accepts the same `label=` / `label_style=` / `attach_to=` /
`parent_id=` arguments as any entity; removing the point removes its label.


## Visual Examples

A scene with two interactive points and a drag handler, exported via
`export_snapshot()` (the actual dragging happens in the live viewer).


In [ ]:
viz = Visualizer(title="Interaction — figure", add_default_axes=False, add_default_grid=False)

# A plain clickable point (low-level API):
pid = viz.add(Point(0, 0, 0), color="#ffaa00", label="click me")
viz.set_interaction(
    pid,
    InteractionConfig(
        enabled=True,
        triggers=[InteractionTrigger(InteractionEventType.CLICK)],
    ),
)

async def on_click(event):
    print("clicked at", event.world_position)

viz.on_interaction(pid, InteractionEventType.CLICK, on_click)

# A draggable ActPoint:
ap = ActPoint(Point(2, 1, 0), drag_mode=DragMode.XY_PLANE)
viz.add(ap, color="#ff4444", label="drag me")

viz.export_snapshot("_output/10_interaction.html", overwrite=True)
print("interaction figure exported")


## Summary

| Task | API |
|---|---|
| Trigger | `InteractionTrigger(event_type, mouse_button, modifiers, drag_mode)` |
| Config | `InteractionConfig(enabled, triggers, throttle_ms, hover_*)` |
| Attach config | `viz.set_interaction(id, config)` |
| Register handler | `viz.on_interaction(id, event_type, async_handler)` |
| Event fields | `event.world_position` / `world_normal` / `world_delta` / `camera` |
| Camera mapping | `camera.project(obj)` / `camera.unproject(obj, depth)` |
| Draggable point | `ActPoint(Point(...), drag_mode=..., handler=...)` |
| Lifecycle | `handler` / `on_drag_start` / `on_drag_end` |
| Constraint | `drag_mode=DragMode.XY_PLANE` |

**Next:** [11 — Animation](../11_animation/).
